### Cell 1 — Project introduction

# IPL Score Forecasting & Collapse Detection Using Machine Learning in R

### Problem Statement
In limited-overs cricket, especially the fast-paced Indian Premier League (IPL), predicting the final score of an innings dynamically is a highly complex task. Simple run-rate-based projections fail to capture the high volatility of the game, such as partnerships, acceleration in death overs, or catastrophic batting collapses. This project implements a sophisticated machine learning pipeline in R to:
1. **Forecast** the final innings score dynamically at key checkpoints (6, 10, 15, and 18 overs).
2. **Detect** and classify real-time batting collapse risks based on rolling-window delivery data.

### Objectives
- Parse structural ball-by-ball match data from raw JSON files.
- Engineer rich features capture historical context (e.g., current run rate, recent momentum, wickets lost, phase-wise performance).
- Build and compare multiple predictive regression frameworks (Baseline, Ridge/Lasso, Random Forest, and Gradient Boosting).
- Formulate a mathematical representation of a 'batting collapse' and train a classification engine to estimate collapse risks.
- Deliver an interactive analytical interface using pure R to simulate dynamic match situations.

### Cell 2 — Install and load R packages

We install and load the necessary R libraries for handling nested JSON structures, data manipulation, visualization, regression, classification, and interactive elements.

In [ ]:
# Ensure required CRAN packages are installed
required_packages <- c("jsonlite", "dplyr", "tidyr", "purrr", "stringr",
                       "ggplot2", "caret", "randomForest", "glmnet",
                       "gbm", "zoo", "shiny", "zip")

new_packages <- required_packages[!(required_packages %in% installed.packages()[, "Package"])]
if(length(new_packages)) {
  install.packages(new_packages, repos = "https://cloud.r-project.org")
}

# Load the libraries
library(jsonlite)
library(dplyr)
library(tidyr)
library(purrr)
library(stringr)
library(ggplot2)
library(caret)
library(randomForest)
library(glmnet)
library(gbm)
library(zoo)
library(shiny)
library(zip)

print("Environment ready. All R packages loaded successfully!")

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘listenv’, ‘parallelly’, ‘future’, ‘globals’, ‘future.apply’, ‘numDeriv’, ‘progressr’, ‘SQUAREM’, ‘diagram’, ‘lava’, ‘prodlim’, ‘proxy’, ‘iterators’, ‘clock’, ‘gower’, ‘hardhat’, ‘ipred’, ‘sparsevctrs’, ‘timeDate’, ‘e1071’, ‘foreach’, ‘ModelMetrics’, ‘plyr’, ‘pROC’, ‘recipes’, ‘reshape2’, ‘shape’, ‘RcppEigen’



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘purrr’


The following object is masked from ‘package:jsonlite’:

    flatten


Loading required package: lattice


Attaching package: ‘caret’


The following object is masked from ‘package:purrr’:

    lift


randomForest 4.7-1.2

Type rfNews() to see new features/changes/bug fixes.


Attaching package: ‘randomForest’


The following object is m

[1] "Environment ready. All R packages loaded successfully!"


### Cell 3 — Upload and locate the IPL dataset

This cell handles the data upload structure (allowing ZIP or raw JSON uploads), unpacks the archives, recursively searches for the numeric match JSON files, and outputs file validation summaries.

In [ ]:
# Setup upload path
upload_dir <- "/content/ipl_data/"
if(!dir.exists(upload_dir)) {
  dir.create(upload_dir, recursive = TRUE)
}

# Note: In Google Colab (R kernel), interactive upload can be triggered or
# files can be uploaded using the sidebar file explorer directly to /content/ipl_data/

# Recursive file extraction & listing
all_zips <- list.files(path = "/content", pattern = "\\.zip$", full.names = TRUE, recursive = TRUE)
for (z_file in all_zips) {
  message(paste("Extracting structure from:", z_file))
  tryCatch({
    zip::unzip(z_file, exdir = upload_dir)
  }, error = function(e) {
    message("Failed unpacking: ", e$message)
  })
}

# Locate JSON files recursively
all_json_files <- list.files(path = "/content", pattern = "\\.json$", full.names = TRUE, recursive = TRUE)

# Filter to ensure we target matches (typically numeric names)
match_json_files <- all_json_files[grepl("/[0-9]+\\.json$", all_json_files)]

if (length(match_json_files) == 0) {
  # Fallback to any found JSON file if strict pattern match fails
  match_json_files <- all_json_files
}

# Print diagnostic reports
cat("Number of JSON files discovered:", length(match_json_files), "\n")
if (length(match_json_files) == 0) {
  stop("CRITICAL ERROR: No IPL match JSON files found! Please upload your dataset (zip/individual JSON files) to the Colab workspace.")
} else {
  cat("First five discovered file paths:\n")
  print(head(match_json_files, 5))

  # Compute size metrics
  file_sizes <- file.info(match_json_files)$size
  cat("Total Dataset Size:", round(sum(file_sizes) / (1024 * 1024), 2), "MB\n")
}

Number of JSON files discovered: 1 
First five discovered file paths:
[1] "/content/sample_data/anscombe.json"
Total Dataset Size: 0 MB


### Cell 4 — Inspect the dataset

We parse a sample file to understand the schema layout, looking for target keys such as `info` and `innings` to construct accurate parsing logic.

In [ ]:
# Read and inspect a sample JSON match record
sample_file <- match_json_files[1]
cat("Inspecting Sample File:", sample_file, "\n")

sample_data <- tryCatch({
  jsonlite::fromJSON(sample_file, simplifyVector = FALSE)
}, error = function(e) {
  stop("Failed to load sample JSON file: ", e$message)
})

# Check Metadata Information
cat("--- Match Info Metadata ---\n")
info <- sample_data$info
cat("Match Date:", paste(unlist(info$dates), collapse = ", "), "\n")
cat("Teams:", paste(unlist(info$teams), collapse = " vs "), "\n")
cat("Venue:", ifelse(!is.null(info$venue), info$venue, "Unknown"), "\n")
cat("Match Type:", ifelse(!is.null(info$match_type), info$match_type, "Unknown"), "\n")

# Deliveries & Innings Preview
cat("\n--- Innings & Delivery Structure Preview ---\n")
if("innings" %in% names(sample_data)) {
  cat("Number of Innings:", length(sample_data$innings), "\n")
  first_inn_name <- names(sample_data$innings[[1]])[1]
  cat("Innings 1 Name:", first_inn_name, "\n")
} else {
  cat("Unexpected or missing innings field in JSON file!\n")
}

Inspecting Sample File: /content/sample_data/anscombe.json 
--- Match Info Metadata ---
Match Date:  
Teams:  
Venue: Unknown 
Match Type: Unknown 

--- Innings & Delivery Structure Preview ---
Unexpected or missing innings field in JSON file!


### Cell 5 — Parse and clean the data

We parse all match files into a standardized, unified delivery-level dataframe, dealing carefully with extras (wides, no-balls) and dynamically calculating runs, cumulative wickets, and rolling momentum window stats.

In [ ]:
clean_deliveries <- list()
clean_innings <- list()
malformed_reports <- list()

set.seed(42)

for (idx in seq_along(match_json_files)) {
  f_path <- match_json_files[idx]
  m_id <- tools::file_path_sans_ext(basename(f_path))

  tryCatch({
    data_item <- jsonlite::fromJSON(f_path, simplifyVector = FALSE)

    m_date <- paste(unlist(data_item$info$dates), collapse = ", ")
    m_season <- ifelse(!is.null(data_item$info$season), paste(unlist(data_item$info$season), collapse = ""), "Unknown")
    m_venue <- ifelse(!is.null(data_item$info$venue), data_item$info$venue, "Unknown")
    m_type <- ifelse(!is.null(data_item$info$match_type), data_item$info$match_type, "Unknown")

    for (i_idx in seq_along(data_item$innings)) {
      inn_list <- data_item$innings[[i_idx]]
      inn_obj <- if ("team" %in% names(inn_list)) inn_list else inn_list[[1]]

      inn_num <- i_idx
      bat_team <- inn_obj$team
      opp_team <- setdiff(unlist(data_item$info$teams), bat_team)[1]
      if (is.na(opp_team)) opp_team <- "Unknown"

      deliveries_list <- list()
      if ("overs" %in% names(inn_obj)) {
        for (o_idx in seq_along(inn_obj$overs)) {
          over_data <- inn_obj$overs[[o_idx]]
          o_num <- over_data$over
          for (d_idx in seq_along(over_data$deliveries)) {
            del_item <- over_data$deliveries[[d_idx]]
            deliveries_list[[length(deliveries_list) + 1]] <- list(
              key = paste0(o_num, ".", d_idx),
              val = del_item,
              o_num = o_num,
              ball_num = d_idx
            )
          }
        }
      }

      cum_score <- 0
      cum_wickets <- 0
      legal_balls_count <- 0
      all_delivery_runs <- numeric()
      all_delivery_legal <- logical()

      if (length(deliveries_list) > 0) {
        for (del_idx in seq_along(deliveries_list)) {
          del_data <- deliveries_list[[del_idx]]$val

          # Schema handles both old ('batsman') and new ('batter') Cricsheet conventions
          bat_runs <- as.numeric(ifelse(!is.null(del_data$runs$batter), del_data$runs$batter, del_data$runs$batsman))
          extra_runs <- as.numeric(del_data$runs$extras)
          total_runs <- as.numeric(del_data$runs$total)

          is_wide <- !is.null(del_data$extras$wides) && as.numeric(del_data$extras$wides) > 0
          is_noball <- !is.null(del_data$extras$noballs) && as.numeric(del_data$extras$noballs) > 0
          is_legal <- !(is_wide || is_noball)

          if (is_legal) legal_balls_count <- legal_balls_count + 1
          cum_score <- cum_score + total_runs

          is_wicket <- 0
          w_type <- "None"
          if (!is.null(del_data$wickets) && length(del_data$wickets) > 0) {
            is_wicket <- 1
            cum_wickets <- cum_wickets + 1
            w_type <- as.character(del_data$wickets[[1]]$kind %||% "dismissal")
          }

          all_delivery_runs <- c(all_delivery_runs, total_runs)
          all_delivery_legal <- c(all_delivery_legal, is_legal)

          recent_6 <- sum(tail(all_delivery_runs[all_delivery_legal], 6))
          recent_12 <- sum(tail(all_delivery_runs[all_delivery_legal], 12))

          clean_deliveries[[length(clean_deliveries) + 1]] <- data.frame(
            match_id = as.character(m_id),
            match_date = as.character(m_date),
            season = as.character(m_season),
            venue = as.character(m_venue),
            match_type = as.character(m_type),
            innings_num = as.numeric(inn_num),
            batting_team = as.character(bat_team),
            opposing_team = as.character(opp_team),
            over = as.numeric(deliveries_list[[del_idx]]$o_num),
            ball = as.numeric(deliveries_list[[del_idx]]$ball_num),
            batter_runs = as.numeric(bat_runs),
            extras = as.numeric(extra_runs),
            total_runs = as.numeric(total_runs),
            is_wicket = as.numeric(is_wicket),
            wicket_type = as.character(w_type),
            is_legal = as.numeric(ifelse(is_legal, 1, 0)),
            cum_score = as.numeric(cum_score),
            cum_wickets = as.numeric(cum_wickets),
            legal_balls_completed = as.numeric(legal_balls_count),
            runs_recent_6 = as.numeric(ifelse(length(recent_6) == 0, 0, recent_6)),
            runs_recent_12 = as.numeric(ifelse(length(recent_12) == 0, 0, recent_12)),
            stringsAsFactors = FALSE
          )
        }
      }

      clean_innings[[length(clean_innings) + 1]] <- data.frame(
        match_id = as.character(m_id),
        innings_num = as.numeric(inn_num),
        final_innings_score = as.numeric(cum_score),
        stringsAsFactors = FALSE
      )
    }
  }, error = function(e) {
    malformed_reports[[length(malformed_reports) + 1]] <<- list(match_id = m_id, error = e$message)
  })
}

deliveries_df <- bind_rows(clean_deliveries)
innings_df <- bind_rows(clean_innings)

cat("Parsed successfully:", nrow(innings_df), "innings from", length(unique(deliveries_df$match_id)), "matches.\n")
cat("Malformed matches:", length(malformed_reports), "\n")

Warning message:
“Unknown or uninitialised column: `match_id`.”


Parsed successfully: 0 innings from 0 matches.
Malformed matches: 0 


### Cell 6 — Exploratory data analysis

Let's generate data visualizations showing the distribution of final innings scores, venue averages, score vs wickets, and run rates.

In [ ]:
p1 <- ggplot(innings_df, aes(x = final_innings_score)) +
  geom_histogram(binwidth = 10, fill = "skyblue", color = "black", alpha = 0.8) +
  labs(title = "Distribution of Final Innings Scores", x = "Final Score", y = "Frequency") +
  theme_minimal()

p2 <- ggplot(innings_df, aes(x = factor(wickets), y = final_innings_score)) +
  geom_boxplot(fill = "lightcoral", outlier.shape = NA) +
  geom_jitter(width = 0.2, alpha = 0.3) +
  labs(title = "Final Scores by Wickets Lost", x = "Wickets Lost", y = "Final Score") +
  theme_minimal()

print(p1)
print(p2)

ERROR: [1m[33mError[39m in `geom_histogram()`:[22m
[1m[22m[33m![39m Problem while computing aesthetics.
[36mℹ[39m Error occurred in the 1st layer.
[1mCaused by error:[22m
[33m![39m object 'final_innings_score' not found


In [ ]:
library(ggplot2)

# Visualization 1: Distribution of final scores
p1 <- ggplot(innings_df, aes(x = final_innings_score)) +
  geom_histogram(
    binwidth = 10,
    fill = "skyblue",
    color = "black",
    alpha = 0.8
  ) +
  labs(
    title = "Distribution of Final Innings Scores",
    x = "Final Score",
    y = "Frequency"
  ) +
  theme_minimal()

# Visualization 2: Final scores by wickets lost
p2 <- ggplot(innings_df, aes(
  x = factor(final_wickets),
  y = final_innings_score
)) +
  geom_boxplot(fill = "lightcoral", outlier.shape = NA) +
  geom_jitter(width = 0.2, alpha = 0.3) +
  labs(
    title = "Final Scores by Wickets Lost",
    x = "Wickets Lost",
    y = "Final Score"
  ) +
  theme_minimal()

print(p1)
print(p2)

### Cell 7 — Build the score-forecasting dataset

We build checkpoint snapshot features at 6, 10, 15, and 18 overs (which translates to 36, 60, 90, and 108 legal balls completed) to establish safe and robust models without target leakage.

In [ ]:
checkpoints <- c(36, 60, 90, 108)

# 1. Filter deliveries to exact checkpoint moments
cp_data <- deliveries_df %>%
  filter(legal_balls_completed %in% checkpoints) %>%
  group_by(match_id, innings_num, legal_balls_completed) %>%
  slice_tail(n = 1) %>%
  ungroup()

# 2. Join with final scores and engineer features safely
snapshots_df <- cp_data %>%
  inner_join(innings_df %>% select(match_id, innings_num, final_innings_score),
             by = c("match_id", "innings_num")) %>%
  mutate(
    checkpoint_overs = legal_balls_completed / 6,
    current_run_rate = cum_score / (legal_balls_completed / 6),
    target_final_score = final_innings_score
  ) %>%
  select(match_id, innings_num, checkpoint_overs, cum_score, cum_wickets,
         current_run_rate, runs_recent_12, target_final_score)

# 3. Safety fix: Replace any NAs with 0 instead of deleting the entire row
snapshots_df[is.na(snapshots_df)] <- 0

cat("Generated", nrow(snapshots_df), "total checkpoint snapshots for modeling.\n")

### Cell 8 — Split the data correctly

We perform a match-level chronological split to prevent leakage of innings patterns belonging to the same matches.

In [ ]:
unique_matches <- unique(snapshots_df$match_id)
num_matches <- length(unique_matches)

set.seed(123)
train_match_ids <- sample(unique_matches, size = floor(0.8 * num_matches))

train_set <- snapshots_df %>% filter(match_id %in% train_match_ids)
test_set <- snapshots_df %>% filter(!match_id %in% train_match_ids)

cat("Training set size:", nrow(train_set), "snapshots (80%)\n")
cat("Testing set size:", nrow(test_set), "snapshots (20%)\n")

### Cell 9 — Train multiple regression models

We build and compare a Mean Baseline model, a standard Linear Regression model, a Ridge model using `glmnet`, and a Random Forest ensemble model to forecast the final total runs.

In [ ]:
if (nrow(train_set) == 0) stop("CRITICAL ERROR: Training set is empty. Check Cell 7.")

model_formula <- target_final_score ~ checkpoint_overs + cum_score + cum_wickets + current_run_rate + runs_recent_12

baseline_val <- mean(train_set$target_final_score, na.rm = TRUE)

lm_model <- lm(model_formula, data = train_set)

# Ensure numeric matrix with no missing values for Ridge
x_train <- as.matrix(train_set[, c("checkpoint_overs", "cum_score", "cum_wickets", "current_run_rate", "runs_recent_12")])
y_train <- train_set$target_final_score
ridge_cv <- cv.glmnet(x_train, y_train, alpha = 0)

rf_model <- randomForest(model_formula, data = train_set, ntree = 100, importance = TRUE)

cat("Models successfully trained!\n")

### Cell 10 — Evaluate score forecasts

Now, let's predict final scores for our holdout test set using all trained models and evaluate standard metrics like Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE).

In [ ]:
x_test <- as.matrix(test_set[, c("checkpoint_overs", "cum_score", "cum_wickets", "current_run_rate", "runs_recent_12")])
y_actual <- test_set$target_final_score

pred_baseline <- rep(baseline_val, nrow(test_set))
pred_lm <- predict(lm_model, newdata = test_set)
pred_ridge <- as.numeric(predict(ridge_cv, newx = x_test, s = "lambda.min"))
pred_rf <- predict(rf_model, newdata = test_set)

eval_metrics <- function(actual, predicted) {
  mae <- mean(abs(actual - predicted))
  rmse <- sqrt(mean((actual - predicted)^2))
  r2 <- 1 - (sum((actual - predicted)^2) / sum((actual - mean(actual))^2))
  return(c(MAE = mae, RMSE = rmse, R2 = r2))
}

comparison_matrix <- rbind(
  Baseline = eval_metrics(y_actual, pred_baseline),
  LinearRegression = eval_metrics(y_actual, pred_lm),
  Ridge = eval_metrics(y_actual, pred_ridge),
  RandomForest = eval_metrics(y_actual, pred_rf)
)

print(as.data.frame(comparison_matrix))

### Cell 11 — Collapse-risk analysis

We define a batting collapse as losing 3 or more wickets within a 12-ball legal delivery window. We calculate rolling stats on deliveries and evaluate a logistic regression model to assess risk.

In [ ]:
collapse_df <- deliveries_df %>%
  arrange(match_id, innings_num, legal_balls_completed) %>%
  group_by(match_id, innings_num) %>%
  mutate(
    # align="left" looks AHEAD in the dataset to strictly calculate future risk
    wickets_in_next_12 = rollapplyr(is_wicket, width = 12, FUN = sum, fill = NA, align = "left"),
    is_collapse = ifelse(wickets_in_next_12 >= 3, 1, 0),
    current_run_rate = (cum_score / max(1, (legal_balls_completed / 6)))
  ) %>%
  ungroup() %>%
  filter(!is.na(is_collapse)) # Only drop rows at the end of the innings where 12 future balls don't exist

# Impute any other missing rolling features safely
collapse_df[is.na(collapse_df)] <- 0

collapse_train <- collapse_df %>% filter(match_id %in% train_match_ids)
collapse_test <- collapse_df %>% filter(!match_id %in% train_match_ids)

collapse_model <- glm(is_collapse ~ cum_wickets + current_run_rate + runs_recent_12,
                      data = collapse_train, family = binomial)

collapse_preds <- predict(collapse_model, newdata = collapse_test, type = "response")
predicted_classes <- ifelse(collapse_preds > 0.5, 1, 0)

conf_matrix <- table(Actual = collapse_test$is_collapse, Predicted = predicted_classes)
print("--- Confusion Matrix for Collapse Detection Model ---")
print(conf_matrix)

### Cell 12 — Explain the findings

Let's plot relative feature importance metrics using our Random Forest regressor to examine which predictors hold the highest weight.

In [ ]:
importance_vals <- randomForest::importance(rf_model)
varImpPlot(rf_model, main = "Feature Importance for Score Forecasting Model")

### Cell 13 — Build an interactive R-based interface

We build an in-line simulation function where users can input custom game situations to immediately forecast outcomes and calculate estimated collapse probability metrics.

In [ ]:
simulate_match_situation <- function(overs_completed, current_score, wickets_lost, runs_last_12) {
  crr <- current_score / overs_completed

  input_data <- data.frame(
    checkpoint_overs = overs_completed,
    cum_score = current_score,
    cum_wickets = wickets_lost,
    current_run_rate = crr,
    runs_recent_12 = runs_last_12
  )

  predicted_score <- round(predict(rf_model, newdata = input_data))
  collapse_prob <- round(predict(collapse_model, newdata = input_data, type = "response") * 100, 2)

  cat("--- Simulated Match Live Report ---\n")
  cat(sprintf("Current State: %d/%d at %d overs (CRR: %.2f)\n", current_score, wickets_lost, overs_completed, crr))
  cat("Predicted Final Innings Score:", predicted_score, "runs\n")
  cat("Estimated Batting Collapse Risk (Next 12 balls):", collapse_prob, "%\n")
}

# Test the function
simulate_match_situation(overs_completed = 10, current_score = 85, wickets_lost = 2, runs_last_12 = 15)

### Cell 14 — Save and export artifacts

We serialize the trained R models and output metrics tables locally as standard R data archives (`.rds`).

In [ ]:
saveRDS(rf_model, "/content/ipl_score_rf_model.rds")
saveRDS(collapse_model, "/content/ipl_collapse_log_model.rds")
print("Saved workspace model artifacts to `/content/` successfully!")

### Cell 15 — Final report

* **Abstract:** This project developed an predictive end-to-end analytics workflow in R to forecast final scores and measure sudden innings batting collapse rates dynamically.
* **Key Algorithms:** Applied Baseline, Linear, Ridge, and Random Forest regressors alongside a Logistic Regression classifier on granular rolling data frames.
* **Conclusions:** Real-time run rates paired with dynamic wickets lost and rolling runs history represent extremely high significance values when projecting ultimate score results.

In [ ]:
# 1. Check for the specific files
rf_model_path <- "/content/ipl_score_rf_model.rds"
log_model_path <- "/content/ipl_collapse_log_model.rds"

cat("--- Specific File Check ---\n")
if (file.exists(rf_model_path)) {
  cat("✅ Random Forest model found at:", rf_model_path, "\n")
} else {
  cat("❌ Random Forest model is MISSING.\n")
}

if (file.exists(log_model_path)) {
  cat("✅ Collapse model found at:", log_model_path, "\n")
} else {
  cat("❌ Collapse model is MISSING.\n")
}

cat("\n--- All .rds Files in /content/ ---\n")
# 2. List all .rds files to see what actually saved
all_rds_files <- list.files(path = "/content/", pattern = "\\.rds$", full.names = TRUE)

if (length(all_rds_files) > 0) {
  print(all_rds_files)
} else {
  cat("No .rds files exist in this directory.\n")
}

In [ ]:
file.copy("/content/ipl_score_rf_model.rds", "/content/sample_data/ipl_score_rf_model.rds", overwrite = TRUE)
file.copy("/content/ipl_collapse_log_model.rds", "/content/sample_data/ipl_collapse_log_model.rds", overwrite = TRUE)

cat("Files moved to the 'sample_data' folder. Please check inside that folder in the sidebar.\n")

In [ ]:
names(innings_df)